# NB10：MCP 伺服器與企業 API 整合
## 讓 AI 安全連接企業系統

---

### 學習目標

| 目標 | 說明 |
|------|------|
| 理解 MCP 協議 | 了解什麼是 Model Context Protocol、為何重要 |
| 建立 MCP 伺服器 | 用 Python 從零實作最小化 MCP 伺服器 |
| 工具暴露 | 透過 MCP 暴露：檔案讀取、資料庫查詢、REST API 呼叫 |
| OAuth 2.0 | 了解企業級 API 認證的 Client Credentials Flow |
| 安全憑證管理 | 在 Agentic 系統中安全處理 Token 與憑證 |
| LLM 整合 | 將 OpenAI gpt-4o-mini 接入 MCP 工具鏈 |

---

### 架構概覽

```
┌─────────────────────────────────────────────────────────────────┐
│                        Enterprise AI System                      │
│                                                                  │
│  ┌──────────────┐    MCP Protocol    ┌──────────────────────┐  │
│  │              │ ◄────────────────► │    MCP Server         │  │
│  │  LLM Agent   │    JSON-RPC 2.0    │  ┌────────────────┐  │  │
│  │ (gpt-4o-mini)│                    │  │  Tool Registry  │  │  │
│  │              │                    │  │  - read_file    │  │  │
│  └──────────────┘                    │  │  - query_db     │  │  │
│         │                            │  │  - call_api     │  │  │
│         │                            │  └────────────────┘  │  │
│         │                            │         │             │  │
│         │                            │  ┌──────▼──────────┐ │  │
│         │                            │  │  Auth Layer     │ │  │
│         │                            │  │  (OAuth 2.0)    │ │  │
│         │                            │  └─────────────────┘ │  │
│         │                            └──────────────────────┘  │
│         │                                      │                │
│         └──────── tool_calls ──────────────────┘                │
└─────────────────────────────────────────────────────────────────┘
```

**使用技術：** OpenAI gpt-4o-mini · Python · 純手工 MCP 伺服器實作 · OAuth 2.0 Mock

---
## Part 0：環境設定

In [ ]:
# 安裝必要套件
# mcp 套件為選用；若未安裝，本 notebook 使用自行實作的 MCP 相容類別
!pip install openai python-dotenv httpx --quiet

In [ ]:
import os
import json
import time
import hashlib
import asyncio
import re
import logging
from datetime import datetime, timedelta
from typing import Any, Callable, Dict, List, Optional
from dataclasses import dataclass, field

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-api-key-here")
client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

# 設定日誌
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

print("環境設定完成")
print(f"使用模型：{MODEL}")

---
## Part 1：MCP 概念介紹

### 1.1 什麼是 Model Context Protocol (MCP)？

**MCP（Model Context Protocol）** 是由 Anthropic 於 2024 年提出的開放標準，定義了 AI 模型如何安全地與外部資料來源及工具互動的統一協議。

```
傳統做法（緊耦合）：
┌─────────┐    hardcoded    ┌──────────────────────────┐
│   LLM   │ ─────────────► │  API / DB / Filesystem   │
│  Agent  │                │  (每個都要寫專屬整合程式碼) │
└─────────┘                └──────────────────────────┘

MCP 做法（鬆耦合）：
┌─────────┐   MCP Protocol  ┌────────────┐   wraps   ┌──────────────┐
│   LLM   │ ──────────────► │ MCP Server │ ─────────► │  Any Backend │
│  Client │ ◄────────────── │ (統一介面) │ ◄───────── │  API/DB/Files│
└─────────┘   JSON-RPC 2.0  └────────────┘            └──────────────┘
```

### 1.2 MCP 核心概念比較

| 概念 | 說明 | 類比 |
|------|------|----- |
| **Tool（工具）** | LLM 可呼叫的函式，有輸入/輸出 schema | REST API 的 POST endpoint |
| **Resource（資源）** | LLM 可讀取的靜態資料（URI 定址） | REST API 的 GET endpoint |
| **Prompt（提示）** | 伺服器提供的預設提示模板 | 可參數化的 System Prompt |
| **Sampling** | 客戶端代理模型推論（進階） | 反向代理 LLM 呼叫 |

### 1.3 MCP vs 直接 Function Calling

| 面向 | 直接 Function Calling | MCP |
|------|----------------------|-----|
| 工具定義位置 | 嵌入在應用程式碼中 | 獨立在 MCP Server |
| 工具複用 | 每個應用重複實作 | 多個 LLM 客戶端共用 |
| 安全邊界 | 無標準邊界 | Server 端統一驗證 |
| 發現機制 | 靜態（手動維護） | 動態（`tools/list` 端點）|
| 企業治理 | 困難 | 集中管理、稽核 |

### 1.4 企業使用情境

```
企業場景：遺留系統整合

┌──────────────────────────────────────────────────────┐
│                   企業內部系統                         │
│                                                      │
│  ┌──────────┐  ┌──────────┐  ┌──────────────────┐   │
│  │  SAP ERP │  │  Oracle  │  │  Legacy REST API  │   │
│  │  (BAPI)  │  │    DB    │  │  (無 OAuth 支援)  │   │
│  └────┬─────┘  └────┬─────┘  └────────┬─────────┘   │
│       │             │                 │              │
│       └─────────────┴─────────────────┘              │
│                         │                            │
│              ┌──────────▼──────────┐                 │
│              │     MCP Server      │                 │
│              │  (統一安全包裝層)    │                 │
│              │  - 認證 / 授權       │                 │
│              │  - 輸入驗證          │                 │
│              │  - 稽核日誌          │                 │
│              └──────────┬──────────┘                 │
│                         │ MCP Protocol               │
└─────────────────────────┼────────────────────────────┘
                          │
              ┌───────────▼───────────┐
              │   AI Agent (LLM)      │
              │   gpt-4o-mini         │
              └───────────────────────┘
```

---
## Part 2：建立最小化 MCP 伺服器

### 2.1 嘗試匯入官方 mcp 套件（選用）

若環境中未安裝 `mcp` 套件，我們將自行實作一個 MCP 相容的伺服器類別。

In [ ]:
# 嘗試匯入官方 mcp 套件
try:
    from mcp.server import Server as OfficialMCPServer
    from mcp.types import Tool as MCPTool
    HAS_MCP_PACKAGE = True
    print("官方 mcp 套件可用")
except ImportError:
    HAS_MCP_PACKAGE = False
    print("官方 mcp 套件未安裝 → 使用自行實作的 MCP 相容伺服器")

### 2.2 自行實作 MCP 相容伺服器類別

依照 MCP 規範（JSON-RPC 2.0 基礎），實作核心的工具登錄、發現與呼叫機制。

In [ ]:
@dataclass
class MCPToolSchema:
    """MCP Tool 的 Schema 定義（符合 MCP 規範）"""
    name: str
    description: str
    input_schema: Dict[str, Any]  # JSON Schema


@dataclass
class MCPToolResult:
    """MCP Tool 呼叫的回傳結果"""
    content: List[Dict[str, Any]]
    is_error: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return {
            "content": self.content,
            "isError": self.is_error
        }


class MinimalMCPServer:
    """
    最小化 MCP 伺服器實作（不依賴官方 mcp 套件）
    
    實作以下 MCP 端點：
    - initialize       : 交換協議版本與能力
    - tools/list       : 列出所有已登錄工具
    - tools/call       : 呼叫指定工具
    """

    MCP_VERSION = "2024-11-05"

    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self._tools: Dict[str, MCPToolSchema] = {}
        self._handlers: Dict[str, Callable] = {}
        self._audit_log: List[Dict] = []
        self._call_counts: Dict[str, int] = {}
        self._rate_limits: Dict[str, int] = {}  # tool_name -> max calls per session

    def register_tool(
        self,
        name: str,
        description: str,
        input_schema: Dict[str, Any],
        handler: Callable,
        rate_limit: int = 100
    ) -> None:
        """登錄一個 MCP Tool"""
        self._tools[name] = MCPToolSchema(
            name=name,
            description=description,
            input_schema=input_schema
        )
        self._handlers[name] = handler
        self._call_counts[name] = 0
        self._rate_limits[name] = rate_limit
        logger.info(f"[MCP Server] 已登錄工具：{name}")

    # ── MCP Protocol Methods ──────────────────────────────────────

    def initialize(self) -> Dict[str, Any]:
        """MCP initialize 回應（交換版本與能力）"""
        return {
            "protocolVersion": self.MCP_VERSION,
            "serverInfo": {
                "name": self.name,
                "version": self.version
            },
            "capabilities": {
                "tools": {"listChanged": False},
                "resources": {},
                "prompts": {}
            }
        }

    def tools_list(self) -> Dict[str, Any]:
        """MCP tools/list 回應"""
        tools = []
        for schema in self._tools.values():
            tools.append({
                "name": schema.name,
                "description": schema.description,
                "inputSchema": schema.input_schema
            })
        return {"tools": tools}

    def tools_call(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """MCP tools/call 執行工具"""
        timestamp = datetime.now().isoformat()

        # 稽核日誌：記錄每次呼叫
        audit_entry = {
            "timestamp": timestamp,
            "tool": tool_name,
            "arguments": arguments,
            "status": None
        }

        try:
            # 1. 工具存在性檢查
            if tool_name not in self._tools:
                raise ValueError(f"未知工具：{tool_name}")

            # 2. 速率限制檢查
            self._call_counts[tool_name] += 1
            if self._call_counts[tool_name] > self._rate_limits[tool_name]:
                raise RuntimeError(f"工具 '{tool_name}' 已超過速率限制 ({self._rate_limits[tool_name]} 次/session)")

            # 3. 輸入驗證（基本）
            self._validate_input(tool_name, arguments)

            # 4. 執行 handler
            result = self._handlers[tool_name](**arguments)

            audit_entry["status"] = "success"
            self._audit_log.append(audit_entry)

            return MCPToolResult(
                content=[{"type": "text", "text": json.dumps(result, ensure_ascii=False)}]
            ).to_dict()

        except Exception as e:
            audit_entry["status"] = f"error: {str(e)}"
            self._audit_log.append(audit_entry)
            return MCPToolResult(
                content=[{"type": "text", "text": f"錯誤：{str(e)}"}],
                is_error=True
            ).to_dict()

    def _validate_input(self, tool_name: str, arguments: Dict[str, Any]) -> None:
        """基本輸入驗證（防止注入攻擊）"""
        schema = self._tools[tool_name].input_schema
        required = schema.get("required", [])
        for field_name in required:
            if field_name not in arguments:
                raise ValueError(f"缺少必填參數：{field_name}")

        # 字串長度限制（防止過長輸入）
        for key, val in arguments.items():
            if isinstance(val, str) and len(val) > 10000:
                raise ValueError(f"參數 '{key}' 超過長度限制（10000字元）")

    def get_audit_log(self) -> List[Dict]:
        """取得稽核日誌"""
        return self._audit_log.copy()

    def handle_request(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """
        統一 JSON-RPC 2.0 請求處理入口
        符合 MCP 規範的訊息格式
        """
        method = request.get("method", "")
        params = request.get("params", {})
        req_id = request.get("id", 1)

        try:
            if method == "initialize":
                result = self.initialize()
            elif method == "tools/list":
                result = self.tools_list()
            elif method == "tools/call":
                result = self.tools_call(
                    tool_name=params.get("name"),
                    arguments=params.get("arguments", {})
                )
            else:
                raise ValueError(f"不支援的方法：{method}")

            return {"jsonrpc": "2.0", "id": req_id, "result": result}

        except Exception as e:
            return {
                "jsonrpc": "2.0",
                "id": req_id,
                "error": {"code": -32600, "message": str(e)}
            }


print("MinimalMCPServer 類別定義完成")

### 2.3 實作三個企業工具的 Mock Handler

In [ ]:
# ── Mock 資料 ──────────────────────────────────────────────────────

MOCK_FILESYSTEM = {
    "/reports/q4_2024.txt": "Q4 2024 營收報告\n總營收：$4.2M\n同比成長：18%\n主力產品：AI 助理授權",
    "/config/database.json": '{"host": "db.internal", "port": 5432, "name": "enterprise_db"}',
    "/hr/headcount.txt": "工程部門：45人\n業務部門：23人\n產品部門：12人\n總人數：80人"
}

MOCK_DATABASE = {
    "SELECT * FROM products WHERE active=1": [
        {"id": 1, "name": "AI Assistant Pro", "price": 299, "stock": 150},
        {"id": 2, "name": "Data Pipeline",    "price": 499, "stock": 80},
        {"id": 3, "name": "Analytics Dashboard","price": 199, "stock": 200}
    ],
    "SELECT SUM(revenue) FROM sales WHERE year=2024": [
        {"total_revenue": 4200000, "currency": "USD"}
    ],
    "SELECT * FROM employees WHERE dept='engineering'": [
        {"id": 101, "name": "Alice Chen", "role": "Senior Engineer"},
        {"id": 102, "name": "Bob Wang",   "role": "ML Engineer"},
        {"id": 103, "name": "Carol Lin",  "role": "DevOps Engineer"}
    ]
}

MOCK_REST_ENDPOINTS = {
    "https://api.internal/weather": {
        "city": "Taipei", "temperature": 28, "humidity": 75, "condition": "Partly Cloudy"
    },
    "https://api.internal/stock/price": {
        "ticker": "ACME", "price": 142.50, "change": "+1.2%", "volume": 1500000
    },
    "https://api.internal/alerts": {
        "alerts": [
            {"level": "warning", "message": "Server CPU > 80%", "time": "10:35"},
            {"level": "info",    "message": "Backup completed",  "time": "03:00"}
        ]
    }
}


# ── Tool Handler 函式 ─────────────────────────────────────────────

def handle_read_file(path: str) -> Dict[str, Any]:
    """
    讀取指定路徑的檔案內容。
    安全限制：只允許讀取 /reports, /config, /hr 目錄。
    """
    # 路徑安全檢查（防止路徑穿越攻擊）
    ALLOWED_PREFIXES = ["/reports/", "/config/", "/hr/"]
    if not any(path.startswith(p) for p in ALLOWED_PREFIXES):
        raise PermissionError(f"存取被拒：'{path}' 不在允許的目錄範圍內")

    if path not in MOCK_FILESYSTEM:
        raise FileNotFoundError(f"檔案不存在：{path}")

    content = MOCK_FILESYSTEM[path]
    return {
        "path": path,
        "content": content,
        "size": len(content),
        "read_at": datetime.now().isoformat()
    }


def handle_query_database(sql: str) -> Dict[str, Any]:
    """
    執行 SQL 查詢（Mock）。
    安全限制：只允許 SELECT 語句。
    """
    # SQL 注入防護：只允許 SELECT
    sql_clean = sql.strip().upper()
    DANGEROUS_KEYWORDS = ["DROP", "DELETE", "INSERT", "UPDATE", "TRUNCATE", "ALTER", "CREATE"]
    if any(kw in sql_clean for kw in DANGEROUS_KEYWORDS):
        raise PermissionError("僅允許 SELECT 查詢，禁止執行 DDL/DML 語句")

    if not sql_clean.startswith("SELECT"):
        raise ValueError("查詢必須以 SELECT 開頭")

    # 模糊匹配 Mock 資料庫
    result = None
    for pattern, data in MOCK_DATABASE.items():
        if sql.strip().lower() == pattern.lower():
            result = data
            break

    if result is None:
        # 找不到精確匹配，回傳空結果
        result = []

    return {
        "sql": sql,
        "rows": result,
        "row_count": len(result),
        "executed_at": datetime.now().isoformat()
    }


def handle_call_rest_api(
    url: str,
    method: str = "GET",
    params: Optional[Dict] = None
) -> Dict[str, Any]:
    """
    呼叫 REST API（Mock）。
    安全限制：只允許呼叫已登錄的內部端點。
    """
    # URL 白名單驗證
    ALLOWED_DOMAINS = ["api.internal", "internal.corp"]
    from urllib.parse import urlparse
    parsed = urlparse(url)
    if parsed.netloc not in ALLOWED_DOMAINS:
        raise PermissionError(f"不允許呼叫外部域名：{parsed.netloc}")

    method = method.upper()
    if method not in ["GET", "POST"]:
        raise ValueError(f"不支援的 HTTP 方法：{method}")

    # 查找 Mock 回應
    base_url = url.split("?")[0]
    response_data = MOCK_REST_ENDPOINTS.get(base_url, {"error": "端點不存在"})

    return {
        "url": url,
        "method": method,
        "status_code": 200 if "error" not in response_data else 404,
        "response": response_data,
        "called_at": datetime.now().isoformat()
    }


print("Mock Handler 函式定義完成")

In [ ]:
# ── 建立 MCP 伺服器實例並登錄工具 ────────────────────────────────

mcp_server = MinimalMCPServer(name="enterprise-mcp-server", version="1.0.0")

# 工具 1：read_file
mcp_server.register_tool(
    name="read_file",
    description="讀取企業內部檔案系統中的指定檔案內容。僅限讀取 /reports、/config、/hr 目錄。",
    input_schema={
        "type": "object",
        "properties": {
            "path": {
                "type": "string",
                "description": "檔案的絕對路徑，例如 /reports/q4_2024.txt"
            }
        },
        "required": ["path"]
    },
    handler=handle_read_file,
    rate_limit=50
)

# 工具 2：query_database
mcp_server.register_tool(
    name="query_database",
    description="對企業資料庫執行 SELECT 查詢。禁止 DDL/DML 操作。",
    input_schema={
        "type": "object",
        "properties": {
            "sql": {
                "type": "string",
                "description": "SQL SELECT 查詢語句"
            }
        },
        "required": ["sql"]
    },
    handler=handle_query_database,
    rate_limit=30
)

# 工具 3：call_rest_api
mcp_server.register_tool(
    name="call_rest_api",
    description="呼叫企業內部 REST API 端點（僅限 api.internal 網域）。",
    input_schema={
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "API 端點 URL"
            },
            "method": {
                "type": "string",
                "enum": ["GET", "POST"],
                "description": "HTTP 方法",
                "default": "GET"
            },
            "params": {
                "type": "object",
                "description": "查詢參數或請求本體"
            }
        },
        "required": ["url"]
    },
    handler=handle_call_rest_api,
    rate_limit=20
)

print("\n=== MCP 伺服器初始化完成 ===")

In [ ]:
# ── 展示 MCP tools/list 回應 ──────────────────────────────────────

init_request = {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}}
init_response = mcp_server.handle_request(init_request)
print("=== MCP initialize 回應 ===")
print(json.dumps(init_response, ensure_ascii=False, indent=2))

print()

list_request = {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
list_response = mcp_server.handle_request(list_request)
print("=== MCP tools/list 回應 ===")
print(json.dumps(list_response, ensure_ascii=False, indent=2))

In [ ]:
# ── 直接測試 tools/call ───────────────────────────────────────────

print("=== 測試 1：read_file ===")
r1 = mcp_server.handle_request({
    "jsonrpc": "2.0", "id": 3, "method": "tools/call",
    "params": {"name": "read_file", "arguments": {"path": "/reports/q4_2024.txt"}}
})
print(json.dumps(r1, ensure_ascii=False, indent=2))

print("\n=== 測試 2：query_database ===")
r2 = mcp_server.handle_request({
    "jsonrpc": "2.0", "id": 4, "method": "tools/call",
    "params": {"name": "query_database", "arguments": {"sql": "SELECT * FROM products WHERE active=1"}}
})
print(json.dumps(r2, ensure_ascii=False, indent=2))

print("\n=== 測試 3：call_rest_api ===")
r3 = mcp_server.handle_request({
    "jsonrpc": "2.0", "id": 5, "method": "tools/call",
    "params": {"name": "call_rest_api", "arguments": {"url": "https://api.internal/alerts", "method": "GET"}}
})
print(json.dumps(r3, ensure_ascii=False, indent=2))

print("\n=== 測試 4：安全邊界測試（路徑穿越攻擊）===")
r4 = mcp_server.handle_request({
    "jsonrpc": "2.0", "id": 6, "method": "tools/call",
    "params": {"name": "read_file", "arguments": {"path": "/etc/passwd"}}
})
print(json.dumps(r4, ensure_ascii=False, indent=2))

---
## Part 3：MCP 客戶端與 LLM 整合

### 3.1 MCP Client 類別

客戶端負責：
1. 從 MCP Server 取得工具清單（`tools/list`）
2. 將 MCP 工具 schema 轉換為 OpenAI function calling 格式
3. 執行 Agentic Loop：LLM 決定工具 → MCP Client 呼叫 → 回傳結果

In [ ]:
class MCPClient:
    """
    MCP 客戶端：橋接 OpenAI tool_calls 與 MCP Server
    """

    def __init__(self, server: MinimalMCPServer, openai_client: OpenAI, model: str):
        self.server = server
        self.openai_client = openai_client
        self.model = model
        self._mcp_tools_cache: Optional[List[Dict]] = None

    def get_mcp_tools(self) -> List[Dict]:
        """從 MCP Server 取得工具清單"""
        response = self.server.tools_list()
        return response["tools"]

    def convert_to_openai_format(self, mcp_tools: List[Dict]) -> List[Dict]:
        """
        將 MCP tool schema 轉換為 OpenAI function calling 格式

        MCP 格式：
          {"name": ..., "description": ..., "inputSchema": {JSON Schema}}

        OpenAI 格式：
          {"type": "function", "function": {"name": ..., "description": ..., "parameters": {JSON Schema}}}
        """
        openai_tools = []
        for tool in mcp_tools:
            openai_tools.append({
                "type": "function",
                "function": {
                    "name": tool["name"],
                    "description": tool["description"],
                    "parameters": tool["inputSchema"]
                }
            })
        return openai_tools

    def call_mcp_tool(self, tool_name: str, arguments: Dict[str, Any]) -> str:
        """呼叫 MCP Server 的工具並回傳文字結果"""
        result = self.server.tools_call(tool_name, arguments)
        if result.get("isError"):
            return f"工具執行錯誤：{result['content'][0]['text']}"
        return result["content"][0]["text"]

    def run_agent(self, user_query: str, system_prompt: str = None, max_turns: int = 5) -> str:
        """
        執行 Agentic Loop：
        1. LLM 收到問題
        2. LLM 決定呼叫哪些工具
        3. 客戶端呼叫 MCP 工具
        4. 結果回傳 LLM
        5. 重複直到 LLM 給出最終回答
        """
        if system_prompt is None:
            system_prompt = (
                "你是一位企業 AI 助理，可以使用 MCP 工具存取企業系統。"
                "請使用這些工具查詢所需資訊，然後用繁體中文給出清楚的回答。"
            )

        # 取得並轉換工具
        mcp_tools = self.get_mcp_tools()
        openai_tools = self.convert_to_openai_format(mcp_tools)

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ]

        print(f"\n{'='*60}")
        print(f"使用者問題：{user_query}")
        print(f"{'='*60}")

        for turn in range(max_turns):
            print(f"\n[回合 {turn+1}] 呼叫 LLM...")

            response = self.openai_client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto"
            )

            msg = response.choices[0].message
            messages.append(msg.model_dump(exclude_unset=False))

            # LLM 決定不再呼叫工具
            if not msg.tool_calls:
                print(f"\n[最終回答]")
                print(msg.content)
                return msg.content

            # 處理工具呼叫
            for tool_call in msg.tool_calls:
                fn_name = tool_call.function.name
                fn_args = json.loads(tool_call.function.arguments)

                print(f"  → 呼叫工具：{fn_name}")
                print(f"    參數：{json.dumps(fn_args, ensure_ascii=False)}")

                tool_result = self.call_mcp_tool(fn_name, fn_args)

                print(f"    結果（前150字）：{tool_result[:150]}..." if len(tool_result) > 150 else f"    結果：{tool_result}")

                # 將工具結果加入訊息
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": tool_result
                })

        return "達到最大回合數，流程中止。"


# 建立客戶端
mcp_client = MCPClient(server=mcp_server, openai_client=client, model=MODEL)

print("MCP 客戶端建立完成")
print("\n轉換後的 OpenAI 工具格式（前 1 個）：")
openai_tools_preview = mcp_client.convert_to_openai_format(mcp_client.get_mcp_tools())
print(json.dumps(openai_tools_preview[0], ensure_ascii=False, indent=2))

### 3.2 執行 Agentic Loop：用 MCP 工具回答業務問題

In [ ]:
# 業務問題 1：查詢 Q4 報告並彙整
answer1 = mcp_client.run_agent(
    user_query="請讀取 Q4 2024 的營收報告，並同時查詢資料庫中的產品清單，給我一份簡短的業務摘要。"
)

In [ ]:
# 業務問題 2：系統監控
answer2 = mcp_client.run_agent(
    user_query="查看目前的系統警報狀況，並告訴我工程部門有幾位員工。"
)

In [ ]:
# 展示稽核日誌（重要安全特性）
print("=== 工具呼叫稽核日誌 ===")
audit_log = mcp_server.get_audit_log()
for i, entry in enumerate(audit_log, 1):
    print(f"\n[{i}] {entry['timestamp']}")
    print(f"    工具：{entry['tool']}")
    print(f"    參數：{json.dumps(entry['arguments'], ensure_ascii=False)}")
    print(f"    狀態：{entry['status']}")

---
## Part 4：OAuth 2.0 企業認證

### 4.1 OAuth 2.0 Client Credentials Flow（機器對機器認證）

企業的服務對服務（server-to-server）認證場景，**不需要使用者介入**：

```
OAuth 2.0 Client Credentials Flow

┌─────────────────┐                    ┌─────────────────────┐
│   MCP Server    │                    │  Authorization      │
│   (Client)      │                    │  Server             │
│                 │  1. POST /token    │  (企業 IdP)         │
│                 │     client_id      │                     │
│                 │     client_secret  │                     │
│                 │     grant_type=    │                     │
│                 │  client_credentials│                     │
│                 │ ─────────────────► │                     │
│                 │                    │  2. 驗證憑證         │
│                 │                    │     產生 JWT Token   │
│                 │ ◄───────────────── │                     │
│                 │  3. access_token   │                     │
│                 │     expires_in     │                     │
│                 │     token_type     │                     │
│                 │                    └─────────────────────┘
│                 │
│  4. Authorization: Bearer <access_token>                   │
│                 │ ─────────────────► ┌─────────────────────┐
│                 │                    │  Resource Server     │
│                 │ ◄───────────────── │  (企業 API)          │
│                 │  5. 受保護資源      └─────────────────────┘
└─────────────────┘
```

### 4.2 各種 OAuth 2.0 Grant Type 比較

| Grant Type | 使用場景 | 是否需要使用者 |
|------------|----------|----------------|
| **Client Credentials** | 服務對服務（MCP 場景） | 否 |
| Authorization Code | 使用者授權應用程式 | 是 |
| Device Code | 無瀏覽器裝置 | 是 |
| Refresh Token | 更新過期 Token | 否（但需初始授權）|

In [ ]:
import base64
import hmac
import secrets
from dataclasses import dataclass


@dataclass
class TokenInfo:
    """OAuth 2.0 Token 資訊"""
    access_token: str
    token_type: str
    expires_in: int       # 秒
    scope: str
    issued_at: datetime = field(default_factory=datetime.now)

    @property
    def is_expired(self) -> bool:
        return datetime.now() > self.issued_at + timedelta(seconds=self.expires_in)

    @property
    def expires_at(self) -> datetime:
        return self.issued_at + timedelta(seconds=self.expires_in)

    def bearer_header(self) -> str:
        return f"Bearer {self.access_token}"


class MockOAuthTokenServer:
    """
    模擬 OAuth 2.0 授權伺服器（進程內 Mock）
    實作 client_credentials grant type
    """

    def __init__(self):
        # 模擬的客戶端憑證資料庫
        self._clients = {
            "mcp-server-prod": {
                "secret": "s3cr3t-pr0d-key-k8s",
                "scopes": ["read:files", "read:database", "call:api"],
                "description": "生產環境 MCP 伺服器"
            },
            "mcp-server-dev": {
                "secret": "dev-key-not-secure",
                "scopes": ["read:files"],
                "description": "開發環境 MCP 伺服器（限縮權限）"
            }
        }
        self._issued_tokens: Dict[str, Dict] = {}  # token -> metadata
        self.TOKEN_TTL = 3600  # 1 小時

    def token_endpoint(
        self,
        client_id: str,
        client_secret: str,
        grant_type: str,
        scope: str = ""
    ) -> Dict[str, Any]:
        """
        POST /oauth/token 端點模擬
        """
        # 驗證 grant_type
        if grant_type != "client_credentials":
            return {"error": "unsupported_grant_type", "error_description": "僅支援 client_credentials"}

        # 驗證 client_id 存在
        if client_id not in self._clients:
            return {"error": "invalid_client", "error_description": "未知的 client_id"}

        client = self._clients[client_id]

        # 驗證 client_secret（使用 constant-time 比較防止 timing attack）
        if not hmac.compare_digest(client["secret"], client_secret):
            return {"error": "invalid_client", "error_description": "client_secret 錯誤"}

        # 驗證 scope
        requested_scopes = scope.split() if scope else client["scopes"]
        allowed_scopes = [s for s in requested_scopes if s in client["scopes"]]

        # 產生 Access Token
        token = secrets.token_urlsafe(32)
        issued_at = datetime.now()

        self._issued_tokens[token] = {
            "client_id": client_id,
            "scopes": allowed_scopes,
            "issued_at": issued_at,
            "expires_at": issued_at + timedelta(seconds=self.TOKEN_TTL)
        }

        return {
            "access_token": token,
            "token_type": "Bearer",
            "expires_in": self.TOKEN_TTL,
            "scope": " ".join(allowed_scopes)
        }

    def validate_token(self, token: str, required_scope: str = None) -> Dict[str, Any]:
        """驗證 Token 有效性（Resource Server 使用）"""
        if token not in self._issued_tokens:
            return {"valid": False, "error": "invalid_token"}

        metadata = self._issued_tokens[token]
        if datetime.now() > metadata["expires_at"]:
            return {"valid": False, "error": "token_expired"}

        if required_scope and required_scope not in metadata["scopes"]:
            return {"valid": False, "error": f"insufficient_scope：需要 {required_scope}"}

        return {
            "valid": True,
            "client_id": metadata["client_id"],
            "scopes": metadata["scopes"],
            "expires_at": metadata["expires_at"].isoformat()
        }


# 啟動 Mock OAuth Server
oauth_server = MockOAuthTokenServer()
print("Mock OAuth 2.0 授權伺服器啟動完成")

In [ ]:
class EnterpriseTokenManager:
    """
    企業級 Token 管理器
    
    功能：
    - 從環境變數讀取憑證（不硬編碼）
    - 自動取得 Token
    - 自動偵測 Token 過期並刷新
    - Token 快取（避免頻繁請求）
    """

    REFRESH_BEFORE_EXPIRY_SECS = 300  # Token 到期前 5 分鐘主動刷新

    def __init__(self, oauth_server: MockOAuthTokenServer):
        self.oauth_server = oauth_server
        self._current_token: Optional[TokenInfo] = None
        self._token_request_count = 0

        # 安全做法：從環境變數讀取，絕不硬編碼
        self._client_id = os.getenv("OAUTH_CLIENT_ID", "mcp-server-prod")
        self._client_secret = os.getenv("OAUTH_CLIENT_SECRET", "s3cr3t-pr0d-key-k8s")
        self._scope = os.getenv("OAUTH_SCOPE", "read:files read:database call:api")

    def _needs_refresh(self) -> bool:
        """判斷是否需要刷新 Token"""
        if self._current_token is None:
            return True
        # 提前刷新（距離到期還有 5 分鐘時）
        time_to_expiry = (self._current_token.expires_at - datetime.now()).total_seconds()
        return time_to_expiry < self.REFRESH_BEFORE_EXPIRY_SECS

    def get_token(self) -> TokenInfo:
        """取得有效 Token（自動刷新）"""
        if self._needs_refresh():
            self._refresh_token()
        return self._current_token

    def _refresh_token(self) -> None:
        """向 OAuth Server 請求新 Token"""
        self._token_request_count += 1
        print(f"  [TokenManager] 請求新 Token（第 {self._token_request_count} 次）...")

        response = self.oauth_server.token_endpoint(
            client_id=self._client_id,
            client_secret=self._client_secret,
            grant_type="client_credentials",
            scope=self._scope
        )

        if "error" in response:
            raise RuntimeError(f"OAuth Token 取得失敗：{response['error']} - {response.get('error_description')}")

        self._current_token = TokenInfo(
            access_token=response["access_token"],
            token_type=response["token_type"],
            expires_in=response["expires_in"],
            scope=response["scope"]
        )
        print(f"  [TokenManager] Token 取得成功，有效期至：{self._current_token.expires_at.strftime('%H:%M:%S')}")

    def get_auth_header(self) -> str:
        """回傳 HTTP Authorization Header 值"""
        return self.get_token().bearer_header()


# ── 示範 Token 取得與驗證流程 ────────────────────────────────────

print("=== OAuth 2.0 Client Credentials Flow 示範 ===")
print()

token_manager = EnterpriseTokenManager(oauth_server)

# 第一次取得 Token
token = token_manager.get_token()
print(f"\nToken Type：{token.token_type}")
print(f"Scope：{token.scope}")
print(f"到期時間：{token.expires_at.strftime('%H:%M:%S')}")
print(f"Authorization Header：{token.bearer_header()[:60]}...")

# 第二次呼叫（快取，不重複請求）
print("\n--- 第二次取得 Token（應使用快取）---")
token2 = token_manager.get_token()
print(f"使用快取 Token：{token.access_token == token2.access_token}")

In [ ]:
# ── Token 驗證（Resource Server 端） ─────────────────────────────

print("=== Token 驗證示範 ===")
print()

# 情境 1：有效 Token + 有效 Scope
valid_token = token_manager.get_token().access_token
result1 = oauth_server.validate_token(valid_token, required_scope="read:database")
print(f"情境 1（有效 Token + 正確 Scope）：")
print(json.dumps(result1, ensure_ascii=False, indent=2))

# 情境 2：有效 Token + 不足的 Scope
result2 = oauth_server.validate_token(valid_token, required_scope="admin:delete")
print(f"\n情境 2（Scope 不足）：")
print(json.dumps(result2, ensure_ascii=False, indent=2))

# 情境 3：偽造 Token
result3 = oauth_server.validate_token("fake-token-123456")
print(f"\n情境 3（偽造 Token）：")
print(json.dumps(result3, ensure_ascii=False, indent=2))

# 情境 4：開發環境 Token（限縮權限）
dev_response = oauth_server.token_endpoint(
    client_id="mcp-server-dev",
    client_secret="dev-key-not-secure",
    grant_type="client_credentials"
)
result4 = oauth_server.validate_token(dev_response["access_token"], required_scope="read:database")
print(f"\n情境 4（開發環境 Token 嘗試存取資料庫）：")
print(json.dumps(result4, ensure_ascii=False, indent=2))

### 4.3 憑證安全管理最佳實踐

```
憑證管理安全等級（由低到高）：

❌ 等級 1（絕對禁止）：硬編碼在原始碼中
   client_secret = "my-secret-key"  # 危險！會進入版本控制

⚠️  等級 2（開發環境可接受）：.env 檔案（不進入版控）
   OAUTH_CLIENT_SECRET=my-secret-key  # .env 檔，加入 .gitignore

✅ 等級 3（生產環境基本要求）：環境變數（由 CI/CD 注入）
   export OAUTH_CLIENT_SECRET="$(vault kv get -field=secret mcp/prod)"

✅ 等級 4（企業生產標準）：Secret Manager
   - AWS Secrets Manager
   - Azure Key Vault
   - HashiCorp Vault
   - GCP Secret Manager

✅ 等級 5（最高安全）：SPIFFE/SPIRE 工作負載身份
   - 自動輪換，零靜態憑證
   - 基於工作負載身份（不需要 client_secret）
```

### Token 範圍最小化原則（Principle of Least Privilege）

```
✅ 正確做法：按功能需求申請最小 scope

# 只讀報表的服務
OAUTH_SCOPE="read:reports"

# 需要讀取資料庫的服務
OAUTH_SCOPE="read:database read:files"

❌ 錯誤做法：申請所有權限
OAUTH_SCOPE="admin:all write:all delete:all"
```

In [ ]:
# ── 整合示範：帶 OAuth Token 的 MCP 工具呼叫 ─────────────────────

class AuthenticatedMCPServer(MinimalMCPServer):
    """
    帶 OAuth 認證的 MCP 伺服器
    每次工具呼叫前驗證 Bearer Token
    """

    def __init__(self, name: str, version: str, oauth_server: MockOAuthTokenServer):
        super().__init__(name, version)
        self.oauth_server = oauth_server
        # 定義每個工具需要的 scope
        self._tool_scopes: Dict[str, str] = {}

    def register_tool(self, name: str, description: str, input_schema: Dict, handler: Callable,
                      rate_limit: int = 100, required_scope: str = None):
        super().register_tool(name, description, input_schema, handler, rate_limit)
        if required_scope:
            self._tool_scopes[name] = required_scope

    def tools_call_authenticated(
        self,
        tool_name: str,
        arguments: Dict[str, Any],
        bearer_token: str
    ) -> Dict[str, Any]:
        """帶認證的工具呼叫"""
        # 取出 Token 字串（去除 'Bearer ' 前綴）
        token = bearer_token.replace("Bearer ", "").strip()

        # 取得此工具需要的 scope
        required_scope = self._tool_scopes.get(tool_name)

        # 驗證 Token
        validation = self.oauth_server.validate_token(token, required_scope)
        if not validation["valid"]:
            return MCPToolResult(
                content=[{"type": "text", "text": f"認證失敗：{validation['error']}"}],
                is_error=True
            ).to_dict()

        # Token 有效，繼續執行工具
        print(f"  [Auth] Token 有效 | client: {validation['client_id']} | scope: {validation['scopes']}")
        return self.tools_call(tool_name, arguments)


# 建立帶認證的伺服器
auth_mcp_server = AuthenticatedMCPServer(
    name="authenticated-mcp-server",
    version="2.0.0",
    oauth_server=oauth_server
)

auth_mcp_server.register_tool(
    name="read_file",
    description="讀取企業檔案",
    input_schema={"type": "object", "properties": {"path": {"type": "string"}}, "required": ["path"]},
    handler=handle_read_file,
    required_scope="read:files"
)
auth_mcp_server.register_tool(
    name="query_database",
    description="查詢資料庫",
    input_schema={"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]},
    handler=handle_query_database,
    required_scope="read:database"
)

print("=== 帶認證的 MCP 工具呼叫示範 ===")
print()

# 取得有效 Token
valid_bearer = token_manager.get_auth_header()

# 呼叫 1：有效 Token
print("呼叫 1：有效 Token")
result = auth_mcp_server.tools_call_authenticated(
    "read_file",
    {"path": "/reports/q4_2024.txt"},
    valid_bearer
)
print(f"結果：{result['content'][0]['text'][:100]}...")

# 呼叫 2：無效 Token
print("\n呼叫 2：無效 Token")
result2 = auth_mcp_server.tools_call_authenticated(
    "read_file",
    {"path": "/reports/q4_2024.txt"},
    "Bearer invalid-token-xyz"
)
print(f"結果：{result2}")

---
## Part 5：安全邊界與生產注意事項

### 5.1 四大安全層次

In [ ]:
# ── 安全層 1：輸入驗證（防止注入攻擊）────────────────────────────

class InputValidator:
    """
    MCP 工具輸入驗證器
    防禦各種注入攻擊
    """

    # SQL 注入模式
    SQL_INJECTION_PATTERNS = [
        r"(--|;|'|\"|\bOR\b|\bAND\b).*?(\bDROP\b|\bDELETE\b|\bINSERT\b)",
        r"UNION\s+SELECT",
        r"\bEXEC\b|\bEXECUTE\b",
        r"xp_cmdshell"
    ]

    # 路徑穿越模式
    PATH_TRAVERSAL_PATTERNS = [
        r"\.\.[\\/]",  # ../
        r"%2e%2e",     # URL 編碼的 ..
        r"\x00"        # Null byte
    ]

    @classmethod
    def check_sql(cls, sql: str) -> None:
        """檢查 SQL 輸入"""
        sql_upper = sql.upper()
        dangerous = ["DROP", "DELETE", "INSERT", "UPDATE", "TRUNCATE",
                     "ALTER", "CREATE", "EXEC", "EXECUTE", "XP_CMDSHELL"]
        for kw in dangerous:
            if kw in sql_upper:
                raise ValueError(f"SQL 包含危險關鍵字：{kw}")

        for pattern in cls.SQL_INJECTION_PATTERNS:
            if re.search(pattern, sql, re.IGNORECASE):
                raise ValueError(f"偵測到潛在 SQL 注入模式")

    @classmethod
    def check_file_path(cls, path: str) -> None:
        """檢查檔案路徑"""
        for pattern in cls.PATH_TRAVERSAL_PATTERNS:
            if re.search(pattern, path, re.IGNORECASE):
                raise ValueError(f"偵測到路徑穿越攻擊")

        # 不允許絕對路徑外的系統目錄
        BLOCKED_PATHS = ["/etc", "/sys", "/proc", "/root", "/home", "/var", "/usr"]
        if any(path.startswith(p) for p in BLOCKED_PATHS):
            raise ValueError(f"存取被拒：禁止存取系統目錄")

    @classmethod
    def check_url(cls, url: str) -> None:
        """檢查 URL（防止 SSRF）"""
        BLOCKED_HOSTS = [
            "localhost", "127.0.0.1", "0.0.0.0",
            "169.254.169.254",  # AWS metadata endpoint
            "metadata.google.internal"  # GCP metadata endpoint
        ]
        from urllib.parse import urlparse
        parsed = urlparse(url)
        if parsed.hostname in BLOCKED_HOSTS:
            raise ValueError(f"SSRF 防護：禁止存取 {parsed.hostname}")


# 測試輸入驗證
print("=== 輸入驗證測試 ===")

test_cases = [
    ("SQL 正常查詢",    lambda: InputValidator.check_sql("SELECT * FROM products")),
    ("SQL 注入攻擊",    lambda: InputValidator.check_sql("SELECT * FROM users; DROP TABLE users--")),
    ("UNION 注入",      lambda: InputValidator.check_sql("SELECT 1 UNION SELECT password FROM admin")),
    ("正常檔案路徑",    lambda: InputValidator.check_file_path("/reports/q4_2024.txt")),
    ("路徑穿越攻擊",    lambda: InputValidator.check_file_path("/reports/../../../etc/passwd")),
    ("正常 API URL",   lambda: InputValidator.check_url("https://api.internal/data")),
    ("SSRF 攻擊",      lambda: InputValidator.check_url("http://169.254.169.254/latest/meta-data/")),
]

for name, test_fn in test_cases:
    try:
        test_fn()
        print(f"  ✓ {name}：通過")
    except (ValueError, PermissionError) as e:
        print(f"  ✗ {name}：攔截 → {e}")

In [ ]:
# ── 安全層 2：速率限制 ────────────────────────────────────────────

class RateLimiter:
    """
    Token Bucket 演算法速率限制
    每個工具有獨立的速率上限
    """

    def __init__(self, max_calls: int, window_seconds: int = 60):
        self.max_calls = max_calls
        self.window_seconds = window_seconds
        self._call_times: List[float] = []

    def allow(self) -> bool:
        """回傳 True 表示允許此次呼叫"""
        now = time.time()
        # 移除超出時間窗口的記錄
        self._call_times = [t for t in self._call_times if now - t < self.window_seconds]
        if len(self._call_times) >= self.max_calls:
            return False
        self._call_times.append(now)
        return True

    @property
    def current_usage(self) -> int:
        now = time.time()
        return len([t for t in self._call_times if now - t < self.window_seconds])


# 展示速率限制
print("=== 速率限制示範（每 60 秒最多 3 次）===")

limiter = RateLimiter(max_calls=3, window_seconds=60)

for i in range(5):
    allowed = limiter.allow()
    status = "允許" if allowed else "拒絕（超過速率限制）"
    print(f"  呼叫 {i+1}：{status}（目前使用量：{limiter.current_usage}/{limiter.max_calls}）")

In [ ]:
# ── 安全層 3：稽核日誌（結構化） ─────────────────────────────────

class AuditLogger:
    """
    結構化稽核日誌器
    生產環境應對接 SIEM 系統（Splunk、ElasticSearch 等）
    """

    def __init__(self):
        self._log: List[Dict] = []

    def log_tool_call(
        self,
        client_id: str,
        tool_name: str,
        arguments: Dict,
        result_status: str,
        duration_ms: float,
        error: str = None
    ) -> None:
        entry = {
            "event_type": "mcp_tool_call",
            "timestamp": datetime.now().isoformat(),
            "client_id": client_id,
            "tool": tool_name,
            # 敏感參數遮蔽（例如 SQL 中的密碼）
            "arguments_hash": hashlib.sha256(
                json.dumps(arguments, sort_keys=True).encode()
            ).hexdigest()[:16],
            "arguments_preview": str(arguments)[:200],
            "status": result_status,
            "duration_ms": round(duration_ms, 2),
            "error": error
        }
        self._log.append(entry)

    def get_summary(self) -> Dict:
        """統計摘要（用於監控儀表板）"""
        total = len(self._log)
        errors = sum(1 for e in self._log if e["status"] == "error")
        by_tool = {}
        for entry in self._log:
            t = entry["tool"]
            by_tool[t] = by_tool.get(t, 0) + 1
        return {
            "total_calls": total,
            "error_count": errors,
            "error_rate": f"{errors/total*100:.1f}%" if total > 0 else "0%",
            "calls_by_tool": by_tool
        }


# 展示稽核日誌
audit = AuditLogger()

# 模擬幾筆呼叫記錄
audit.log_tool_call("mcp-server-prod", "read_file",     {"path": "/reports/q4.txt"},  "success", 12.3)
audit.log_tool_call("mcp-server-prod", "query_database",{"sql": "SELECT * FROM products"},"success", 45.1)
audit.log_tool_call("mcp-server-prod", "read_file",     {"path": "/etc/passwd"},       "error",   2.1, "PermissionError")
audit.log_tool_call("mcp-server-dev",  "call_rest_api", {"url": "https://api.internal/alerts"},"success", 89.5)

print("=== 稽核日誌記錄 ===")
for entry in audit._log:
    print(json.dumps(entry, ensure_ascii=False, indent=2))

print("\n=== 稽核摘要 ===")
print(json.dumps(audit.get_summary(), ensure_ascii=False, indent=2))

In [ ]:
# ── 安全層 4：工具沙箱概念展示 ────────────────────────────────────

print("=== 工具沙箱設計模式 ===")
print("""
生產環境工具沙箱策略：

1. 低風險工具（讀取型）
   - 直接在 MCP Server 進程中執行
   - 範例：read_file, query_database (SELECT)

2. 中風險工具（外部 API 呼叫）
   - 在獨立 Thread 中執行，設定 Timeout
   - 限制網路存取範圍（白名單）
   - 範例：call_rest_api

3. 高風險工具（程式碼執行）
   - 使用 Docker 容器隔離
   - 設定 CPU/Memory 上限
   - 不允許網路存取
   - 範例：execute_python, run_shell

Python 實作方式：
""")

import concurrent.futures
import signal

def run_with_timeout(fn: Callable, timeout_secs: float, **kwargs) -> Any:
    """
    在獨立 Thread 中執行函式，超時自動中止
    （生產環境中改用 Docker 容器以獲得更強隔離）
    """
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(fn, **kwargs)
        try:
            result = future.result(timeout=timeout_secs)
            return {"status": "success", "result": result}
        except concurrent.futures.TimeoutError:
            return {"status": "timeout", "error": f"執行超過 {timeout_secs} 秒"}
        except Exception as e:
            return {"status": "error", "error": str(e)}


# 測試 Timeout 機制
import time

def fast_task():
    time.sleep(0.1)
    return "快速任務完成"

def slow_task():
    time.sleep(5)  # 模擬超時
    return "慢速任務完成"

print("測試 1（正常執行）：")
r1 = run_with_timeout(fast_task, timeout_secs=2.0)
print(f"  {r1}")

print("\n測試 2（執行超時）：")
r2 = run_with_timeout(slow_task, timeout_secs=1.0)
print(f"  {r2}")

### 5.2 生產環境 Checklist

```
MCP Server 生產上線 Checklist

認證與授權
□ 所有工具呼叫都需要有效的 OAuth 2.0 Bearer Token
□ 每個工具設定最小必要 scope
□ Token 驗證使用 JWT 簽章（非僅查資料庫）
□ client_secret 儲存在 Secret Manager（非環境變數）

輸入驗證
□ 所有字串輸入都有長度上限
□ SQL 查詢僅允許 SELECT（parameterized queries）
□ 檔案路徑白名單驗證
□ URL 白名單（防 SSRF）
□ 禁止 Null byte 和特殊字元注入

速率限制
□ 全域速率限制（保護後端系統）
□ 每個客戶端獨立速率限制
□ 每個工具獨立速率限制
□ 超限時回傳 429 Too Many Requests

稽核與監控
□ 每次工具呼叫寫入不可竄改的稽核日誌
□ 日誌包含：時間、客戶端、工具、參數摘要、結果
□ 異常模式自動告警（例如：短時間大量失敗）
□ 對接 SIEM 系統

網路隔離
□ MCP Server 部署在內部網路，不對外開放
□ 使用 mTLS 進行服務間通訊
□ 工具執行環境與 MCP Server 隔離
```

---
## Part 6：面試 Q&A

### Q1：MCP 與直接 Function Calling 的核心差異是什麼？在什麼情況下應該使用 MCP？

**A1：**

**核心差異：**

| 面向 | Function Calling | MCP |
|------|-----------------|-----|
| 工具定義位置 | 嵌入在 LLM 應用程式碼中 | 獨立的 MCP Server 進程 |
| 工具複用 | 每個應用重複實作 | 多個 Agent/LLM 共用同一 Server |
| 安全邊界 | 無標準邊界，工具直接暴露 | Server 端統一做認證、驗證、稽核 |
| 工具發現 | 靜態定義，手動維護 | 動態 `tools/list`，可熱更新 |
| 企業治理 | 分散、難以管控 | 集中管理，符合合規需求 |

**何時使用 MCP：**

1. **多個 Agent 共用工具**：例如 10 個不同 Agent 都需要查詢同一個資料庫，透過 MCP Server 統一管理，避免重複實作。
2. **需要集中安全控管**：企業合規要求所有外部系統存取必須有稽核日誌、認證授權。
3. **包裝遺留系統（Legacy API）**：把老舊的 SOAP API、BAPI、無文件的 REST API 包裝成標準 MCP 工具。
4. **工具需要獨立部署和版本控制**：工具的業務邏輯複雜，需要獨立測試、部署、回滾。

**不需要 MCP 的情況：** 單一 Agent、工具簡單且不共用、快速原型開發時，直接 Function Calling 更輕量。

### Q2：請解釋 OAuth 2.0 的 Client Credentials Flow，以及為什麼 Agentic 系統適合使用這個流程？

**A2：**

**Client Credentials Flow 運作步驟：**

```
1. MCP Server（Client）持有 client_id 和 client_secret
2. POST /oauth/token
   Body: grant_type=client_credentials
         &client_id=mcp-server-prod
         &client_secret=...
         &scope=read:database
3. Authorization Server 驗證憑證，回傳 access_token（JWT）
4. MCP Server 在每次 API 呼叫時附上 Authorization: Bearer <token>
5. Token 到期前主動刷新（避免服務中斷）
```

**為什麼 Agentic 系統適合使用 Client Credentials：**

1. **無需使用者介入**：Agent 是自動化系統，不可能彈出登入頁面讓使用者輸入密碼。Client Credentials 正是為機器對機器（M2M）場景設計。

2. **可精確控制 Scope**：可以為不同 Agent 設定不同權限，例如「報表 Agent」只有 `read:reports`，不能存取 HR 資料。

3. **Token 自動輪換**：Client Credentials 的 Token 有到期時間（通常 1 小時），定期輪換比長期憑證更安全。如果 Token 洩漏，影響範圍有限。

4. **稽核可追蹤**：每個 Token 都綁定 `client_id`，Authorization Server 可以記錄哪個 Agent 在何時存取了哪些資源。

**注意事項：** `client_secret` 必須儲存在 Secret Manager（如 AWS Secrets Manager）中，絕不能硬編碼在程式碼或放入版本控制。

### Q3：設計一個 MCP 工具時，應該考慮哪些安全威脅？如何防禦？

**A3：**

**五大安全威脅與防禦策略：**

**1. Prompt Injection → Tool Argument Injection**
- **威脅**：攻擊者在使用者輸入中嵌入惡意指令，誘導 LLM 傳入危險參數。
- **防禦**：工具本身不信任 LLM 傳入的任何參數，全部做伺服器端驗證（白名單、長度限制、pattern 匹配）。

**2. SQL Injection**
- **威脅**：`query_database` 工具收到 `SELECT * FROM users; DROP TABLE users--`。
- **防禦**：只允許 SELECT、使用 Parameterized Queries、禁止危險關鍵字。

**3. Path Traversal（路徑穿越）**
- **威脅**：`read_file` 收到 `../../etc/passwd` 嘗試讀取系統檔案。
- **防禦**：路徑白名單（只允許特定目錄）、禁止 `..` 字元、使用 `os.path.realpath` 正規化路徑後再比對。

**4. SSRF（Server-Side Request Forgery）**
- **威脅**：`call_rest_api` 工具被要求呼叫 `http://169.254.169.254`（AWS Metadata API），竊取雲端憑證。
- **防禦**：URL 白名單（只允許 `*.internal` 域名）、禁止 Loopback 和 Link-Local IP。

**5. Resource Exhaustion（資源耗盡）**
- **威脅**：LLM 進入無限迴圈，反覆呼叫工具導致後端 DB 過載。
- **防禦**：每個工具設定速率限制（Token Bucket 演算法）、Agentic Loop 設定最大回合數、工具執行設定 Timeout。

**核心原則：** 永遠在 Server 端驗證，不依賴 LLM 自我約束。

### Q4：企業想把一套舊的 SOAP API 整合到 AI Agent 系統中，你會如何設計整合架構？

**A4：**

**架構設計（由外到內）：**

```
Layer 5：AI Agent（gpt-4o-mini）
         ↕ MCP Protocol（JSON-RPC 2.0）
Layer 4：MCP Server（Python FastAPI / FastMCP）
         - 工具：query_erp(operation, params)
         - OAuth 2.0 Bearer Token 驗證
         - 輸入驗證、速率限制、稽核日誌
         ↕ REST 呼叫
Layer 3：SOAP → REST 轉換層（Adapter）
         - 使用 zeep（Python SOAP 客戶端）
         - 將 WSDL 操作對應為 REST 端點
         - 處理 XML 序列化/反序列化
         - 快取高頻查詢結果（Redis）
         ↕ SOAP（XML over HTTP）
Layer 2：企業防火牆（限制來源 IP）
         ↕
Layer 1：遺留 SOAP API（SAP / Oracle 等）
```

**關鍵設計決策：**

1. **不讓 LLM 直接呼叫 SOAP**：LLM 生成的 XML 難以驗證，透過 MCP 中間層可以精確控制哪些 WSDL 操作可以被呼叫。

2. **工具語意化設計**：不直接暴露 SOAP 操作名稱，而是設計語意清晰的工具，例如 `get_customer_orders(customer_id)`，內部再映射到 `CustomerService.GetOrders` WSDL 方法。

3. **回應格式標準化**：SOAP 回應是複雜的巢狀 XML，MCP Server 的 Adapter 層將其轉換為扁平 JSON，方便 LLM 理解。

4. **錯誤處理**：SOAP Fault 轉換為 MCP 錯誤格式，並記錄詳細錯誤上下文到稽核日誌。

5. **漸進式整合**：先從最常用的 3-5 個操作開始暴露，驗證流程後再逐步擴展，避免一次性整合導致風險過高。

In [ ]:
# ── 本 Notebook 學習重點回顧 ──────────────────────────────────────

print("""
╔══════════════════════════════════════════════════════════════╗
║         NB10 學習重點回顧                                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  MCP 協議                                                    ║
║  ├─ JSON-RPC 2.0 基礎協議                                    ║
║  ├─ 三大原語：Tool / Resource / Prompt                       ║
║  └─ initialize → tools/list → tools/call 流程               ║
║                                                              ║
║  MCP Server 實作                                             ║
║  ├─ 工具登錄（register_tool + JSON Schema）                  ║
║  ├─ 統一入口（handle_request）                               ║
║  └─ 三個企業工具：read_file / query_db / call_rest_api       ║
║                                                              ║
║  LLM 整合                                                    ║
║  ├─ MCP schema → OpenAI tools 格式轉換                       ║
║  ├─ Agentic Loop（最多 N 回合）                              ║
║  └─ tool_calls → MCP tools_call → 回傳結果                  ║
║                                                              ║
║  OAuth 2.0                                                   ║
║  ├─ Client Credentials Flow（機器對機器）                    ║
║  ├─ Token 管理：自動刷新、提前刷新                           ║
║  └─ Scope 最小化原則                                         ║
║                                                              ║
║  安全四層                                                    ║
║  ├─ 輸入驗證（SQL注入、路徑穿越、SSRF）                      ║
║  ├─ 速率限制（Token Bucket）                                 ║
║  ├─ 稽核日誌（結構化、不可竄改）                             ║
║  └─ 工具沙箱（Timeout、Container 隔離）                      ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")